<a href="https://colab.research.google.com/github/Nipun1a/Credit_predict/blob/main/Loan_approval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Importing the Dependencies



In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder


In [10]:
#importing the dataset
loan_dataset = pd.read_csv('/content/loan_data.csv')

In [11]:
#printing the first 5 columns
loan_dataset.head()


,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,3.0,561,No,1
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,2.0,504,Yes,0
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,3.0,635,No,1
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,2.0,675,No,1
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,4.0,586,No,1


In [12]:
#print the last 5 columns
loan_dataset.tail()

,person_age,person_gender,person_education,person_income,person_emp_exp,person_home_ownership,loan_amnt,loan_intent,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,loan_status
44995,27.0,male,Associate,47971.0,6,RENT,15000.0,MEDICAL,15.66,0.31,3.0,645,No,1
44996,37.0,female,Associate,65800.0,17,RENT,9000.0,HOMEIMPROVEMENT,14.07,0.14,11.0,621,No,1
44997,33.0,male,Associate,56942.0,7,RENT,2771.0,DEBTCONSOLIDATION,10.02,0.05,10.0,668,No,1
44998,29.0,male,Bachelor,33164.0,4,RENT,12000.0,EDUCATION,13.23,0.36,6.0,604,No,1
44999,24.0,male,High School,51609.0,1,RENT,6665.0,DEBTCONSOLIDATION,17.05,0.13,3.0,628,No,1


In [13]:
#checking the null values
check = loan_dataset.isnull().sum()
print(check)
#there is no null values so we can proceed further work

person_age                        0
person_gender                     0
person_education                  0
person_income                     0
person_emp_exp                    0
person_home_ownership             0
loan_amnt                         0
loan_intent                       0
loan_int_rate                     0
loan_percent_income               0
cb_person_cred_hist_length        0
credit_score                      0
previous_loan_defaults_on_file    0
loan_status                       0
dtype: int64


In [14]:
X = loan_dataset.drop('loan_status', axis=1)
y = loan_dataset['loan_status']

Now, let's identify and encode the categorical features. We'll use `LabelEncoder` for binary categories and `OneHotEncoder` for multi-class categories.

In [15]:
# Identify categorical and numerical features
categorical_features = X.select_dtypes(include=['object']).columns
numerical_features = X.select_dtypes(include=['int64', 'float64']).columns

print(f"Categorical Features: {list(categorical_features)}")
print(f"Numerical Features: {list(numerical_features)}")

Categorical Features: ['person_gender', 'person_education', 'person_home_ownership', 'loan_intent', 'previous_loan_defaults_on_file']
Numerical Features: ['person_age', 'person_income', 'person_emp_exp', 'loan_amnt', 'loan_int_rate', 'loan_percent_income', 'cb_person_cred_hist_length', 'credit_score']


In [16]:
# Apply Label Encoding to binary categorical features
le = LabelEncoder()
X['person_gender'] = le.fit_transform(X['person_gender'])
X['previous_loan_defaults_on_file'] = le.fit_transform(X['previous_loan_defaults_on_file'])

# Display the mapping for clarity
print(f"Person Gender mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(f"Previous Loan Defaults mapping: {dict(zip(le.classes_, le.transform(le.classes_)))}")

Person Gender mapping: {'No': np.int64(0), 'Yes': np.int64(1)}
Previous Loan Defaults mapping: {'No': np.int64(0), 'Yes': np.int64(1)}


In [17]:
# Apply One-Hot Encoding to multi-class categorical features
X = pd.get_dummies(X, columns=['person_education', 'person_home_ownership', 'loan_intent'], drop_first=True)

# Display the first few rows of the preprocessed data
display(X.head())

,person_age,person_gender,person_income,person_emp_exp,loan_amnt,loan_int_rate,loan_percent_income,cb_person_cred_hist_length,credit_score,previous_loan_defaults_on_file,...,person_education_High School,person_education_Master,person_home_ownership_OTHER,person_home_ownership_OWN,person_home_ownership_RENT,loan_intent_EDUCATION,loan_intent_HOMEIMPROVEMENT,loan_intent_MEDICAL,loan_intent_PERSONAL,loan_intent_VENTURE
0,22.0,0,71948.0,0,35000.0,16.02,0.49,3.0,561,0,...,False,True,False,False,True,False,False,False,True,False
1,21.0,0,12282.0,0,1000.0,11.14,0.08,2.0,504,1,...,True,False,False,True,False,True,False,False,False,False
2,25.0,0,12438.0,3,5500.0,12.87,0.44,3.0,635,0,...,True,False,False,False,False,False,False,True,False,False
3,23.0,0,79753.0,0,35000.0,15.23,0.44,2.0,675,0,...,False,False,False,False,True,False,False,True,False,False
4,24.0,1,66135.0,1,35000.0,14.27,0.53,4.0,586,0,...,False,True,False,False,True,False,False,True,False,False


Now that the data is preprocessed, let's split it into training and testing sets.

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

X_train shape: (36000, 22)
X_test shape: (9000, 22)
y_train shape: (36000,)
y_test shape: (9000,)


Let's train and evaluate our models: Logistic Regression, Random Forest, and XGBoost.

In [19]:
# 1. Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

log_reg = LogisticRegression(random_state=42, solver='liblinear')
log_reg.fit(X_train, y_train)

y_pred_lr = log_reg.predict(X_test)

print("Logistic Regression Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_lr))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_lr))
print("\n" + "="*50 + "\n")

Logistic Regression Performance:
Accuracy: 0.7942
Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.99      0.88      7000
           1       0.72      0.12      0.21      2000

    accuracy                           0.79      9000
   macro avg       0.76      0.55      0.55      9000
weighted avg       0.78      0.79      0.73      9000

Confusion Matrix:
[[6902   98]
 [1754  246]]




In [20]:
# 2. Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(random_state=42)
rf_clf.fit(X_train, y_train)

y_pred_rf = rf_clf.predict(X_test)

print("Random Forest Classifier Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))
print("\n" + "="*50 + "\n")

Random Forest Classifier Performance:
Accuracy: 0.9278
Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      7000
           1       0.89      0.77      0.83      2000

    accuracy                           0.93      9000
   macro avg       0.91      0.87      0.89      9000
weighted avg       0.93      0.93      0.93      9000

Confusion Matrix:
[[6817  183]
 [ 467 1533]]




In [22]:
# 3. XGBoost Classifier
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(random_state=42, eval_metric='logloss') # Removed use_label_encoder
xgb_clf.fit(X_train, y_train)

y_pred_xgb = xgb_clf.predict(X_test)

print("XGBoost Classifier Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_xgb))
print("\n" + "="*50 + "\n")

XGBoost Classifier Performance:
Accuracy: 0.9361
Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.97      0.96      7000
           1       0.89      0.81      0.85      2000

    accuracy                           0.94      9000
   macro avg       0.92      0.89      0.90      9000
weighted avg       0.94      0.94      0.94      9000

Confusion Matrix:
[[6798  202]
 [ 373 1627]]




In [24]:
import joblib

# Save the best model (XGBoost in this case)
joblib.dump(xgb_clf, 'model.pkl')

print("Model 'model.pkl' saved successfully!")

Model 'model.pkl' saved successfully!


Now you can download the `model.pkl` file.

### Model Comparison Summary

Based on the performance metrics above, you can compare the accuracy, precision, recall, and F1-score of each model to determine which one performs best for this dataset.

In [25]:
from google.colab import files
files.download('model.pkl')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>